<div style="background:linear-gradient(90deg,#28a745,#6f42c1); color:white; padding:20px 32px; border-radius:8px; width:97%;">

# 📋 Lesson 1 · 本节精华

</div>

<small>对应主课:[1_a_🔥_todo.ipynb](./1_a_🔥_todo.ipynb)</small>

## 🎯 一句话定位

> 给 agent 加 **TODO 列表** 作为"长任务导航工具",用 **Manus 风格的 recitation(背诵)模式**对抗 **context rot**(上下文衰减)。
>
> 🧠 **本质 = 让 LLM 给未来的自己写笔记**。

## 📚 你应该学到的 7 件事

| # | 概念 | 你应该记住的一句话 |
|---|---|---|
| 1 | 🌫️ **Context rot** | context 一长,LLM 注意力衰减、跑题、忘记目标(Manus 平均每任务 50 次工具调用) |
| 2 | 🎯 **Recitation 模式** | 周期性"复述"目标 → 把计划推到 context **末尾** → 注意力被拉回 |
| 3 | 📦 **`DeepAgentState`** | 在 `AgentState` 上加 `todos: list[Todo]` 和 `files: dict` 两个字段 |
| 4 | ✍️ **`write_todos`** | LLM 给出完整列表 → 工具**覆盖式**写入 state(为何覆盖见下方)|
| 5 | 📖 **`read_todos`** | 从 state 读最新 todos → 塞回 `messages` 末尾(自我提示) |
| 6 | 🚦 **状态机** | `pending` → `in_progress` → `completed`,**同时最多 1 个 in_progress** |
| 7 | 📜 **TODO_USAGE_INSTRUCTIONS** | 5 步循环:**写 → 做 → 读 → 反思 → 改状态 → 做下一个** |

## 🧠 核心心智模型:Recitation 解决什么问题?

**问题画像**:agent 跑 20 轮以后,messages 长这样:

```
[0]  Human:    "调研 MCP"
[1]  AI:       write_todos(["搜MCP", "总结", "对比"])
[2]  Tool:     "Updated todo list to [...]"          ← 🎯 计划在这里!
[3]  AI:       web_search(...)
[4]  Tool:     <长长一段搜索结果>
[5]  AI:       web_search(...)
[6]  Tool:     <又长又乱的结果>
...                                                   ⚠️ 15 轮工具调用后
[31] AI:       ???                                    ← 此刻该干啥?计划已被埋在最上面
```

**注意力衰减**:计划离当前位置太远 → LLM 容易**跑题 / 忘记 / 重复**。

**解药 = 主动调 `read_todos`**:

```
[32] AI:       read_todos()
[33] Tool:     "Current TODO:                         ← 🔥 计划重新出现在末尾!
                1. ✅ 搜MCP (completed)
                2. 🔄 总结  (in_progress)
                3. ⏳ 对比  (pending)"
[34] AI:       ...                                    ← LLM 注意力被强制拉回
```

> 🔑 **核心洞察**:**`read_todos` 不是给程序员读的,是给 LLM 自己读的**。它从权威存储里取出 todos,**塞回 messages 末尾**,这就是"recitation"。

## 💎 3 个容易忽视的设计精髓

### 1️⃣ `todos` 字段为何**没有自定义 reducer**?

默认 reducer = **整体覆盖**。这看似危险(怕丢失历史),但**对 TODO 列表来说是故意的**:

- ✅ Manus 风格要求 LLM **每次重写整个列表** —— 重写过程本身就是"反思"的机会
- ✅ 状态信息少(只有 status),覆盖式更新简单可靠
- ❌ 如果用合并 reducer,LLM 可能漏改 status,留下脏数据

### 2️⃣ `read_todos` 为什么从 state 读,而不从 messages 翻?

messages 里可能有**多条** "Updated todo list to [...]"(每次 write 都加一条)。它们都是历史快照,LLM 不知道哪个是当前的。**直接读 `state["todos"]` 永远拿到最新版**,而 messages 是"聊天记录",state 是"数据库"。

### 3️⃣ `read_todos` 的真正意图 = **自我提示工具**

普通工具是为了**获取外部信息**(web_search、calculator 等);`read_todos` **不获取任何新信息** —— 它读的就是 agent **自己几步前写下的东西**。

它的真正作用:**让 LLM 把自己的计划"复制粘贴"到当前对话末尾,刷新自己的注意力**。这是 deep agent 最反直觉但最关键的一类工具。

## 🌍 真实产品对应 + 一句话

| 你学的 | 现实里的实现 |
|---|---|
| 📝 TODO recitation | **Claude Code 的 `TodoWrite` 工具**,几乎一对一映射 |
| 🎯 Manus 模式来源 | [Manus 团队博客](https://manus.im/blog/Context-Engineering-for-AI-Agents-Lessons-from-Building-Manus),已成生产标配 |
| 🚦 单一 in_progress | Cursor Agent / ChatGPT Agent 都遵循同一规则 |

## ✨ 一句话带走

<div style="background:#e7f7e9; padding:10px 24px; border-left:4px solid #28a745; border-radius:4px; width:97%;">

> 🔑 **TODO 列表 ≠ 普通数据,而是"控制 LLM 注意力的开关"**。
>
> 写 todo 是规划,读 todo 是"逼自己复述目标"。这一节是后续所有"文件系统 / 子 agent"的**思想基础**:**把 LLM 注意力当成稀缺资源来管理**,而不是依赖它自己记住一切。

</div>